In [1]:
%pip install openai

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
from openai import OpenAI
from getpass import getpass

BASE_URL = "http://gpustack.cds.utsa.edu/v1"
API_KEY = getpass("Enter your GPUStack API key: ")

client = OpenAI(
    base_url=BASE_URL,
    api_key=API_KEY
)

print("Client configured.")

Enter your GPUStack API key:  ········


Client configured.


In [3]:
models = client.models.list().data

for i, model in enumerate(models):
    print(f"{i}: {model.id}")

0: qwen3-coder-next
1: qwen3.5-122b-a10b
2: qwen3-vl-embedding-8b
3: qwen3-vl-32b-instruct


In [4]:
MODEL = "qwen3-coder-next"

print("Connected. Model:", MODEL)

Connected. Model: qwen3-coder-next


In [5]:
import time

def ask_model(prompt: str, temperature: float = 1.0, retries: int = 2) -> str:
    """Send one prompt to Qwen Coder and return its text reply."""
    for attempt in range(retries + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
                top_p=0.95,
            )
            return resp.choices[0].message.content
        except Exception:
            if attempt == retries:
                raise
            time.sleep(1.5 * (attempt + 1))

print(ask_model("Say hello in one short sentence."))

Hello! 😊


In [6]:
question = (
    "I am writing a Python function triage_log(line) that asks an LLM to classify a "
    "security log line and returns a dataclass with label and reason. Suggest a robust "
    "prompt format and a defensive way to parse the reply. Keep it concise."
)

print(ask_model(question))

Here’s a concise, robust approach:

**Prompt format:**
```python
PROMPT = """Classify this security log line.

Log: "{log_line}"

Respond *strictly* in JSON format:
{{
  "label": "MALICIOUS"|"SUSPICIOUS"|"NORMAL"|"UNKNOWN",
  "reason": "Brief explanation (≤ 30 words)"
}}
"""
```

**Parsing with defensive handling:**
```python
import dataclasses
import json
import re

@dataclasses.dataclass
class TriageResult:
    label: str
    reason: str

def triage_log(line: str) -> TriageResult:
    try:
        # Assume `llm_call(prompt)` returns raw LLM text
        response = llm_call(PROMPT.format(log_line=line))
        # Extract JSON (handles extra text before/after)
        json_match = re.search(r'\{(?:[^{}]|(?R))*\}', response, re.DOTALL)
        if not json_match:
            raise ValueError("No JSON found in response")
        data = json.loads(json_match.group())
        
        label = data.get("label", "").upper()
        if label not in {"MALICIOUS", "SUSPICIOUS", "NORMAL", "UNKNOW

In [7]:
from dataclasses import dataclass
from enum import Enum
import re

class Label(str, Enum):
    BENIGN = "BENIGN"
    SUSPICIOUS = "SUSPICIOUS"
    MALICIOUS = "MALICIOUS"
    UNKNOWN = "UNKNOWN"

@dataclass
class Triage:
    line: str
    label: Label
    reason: str

LABEL_RE = re.compile(
    r"^LABEL:\s*(\w+)",
    re.IGNORECASE | re.MULTILINE
)

REASON_RE = re.compile(
    r"^REASON:\s*(.+)",
    re.IGNORECASE | re.MULTILINE
)

In [8]:
def triage_log(line: str) -> Triage:
    prompt = (
        "You are a security log triage assistant.\n"
        "Classify the log line as BENIGN, SUSPICIOUS, or MALICIOUS and give a one "
        "sentence reason.\n"
        "Reply in EXACTLY this format:\n"
        "LABEL: <BENIGN|SUSPICIOUS|MALICIOUS>\n"
        "REASON: <one sentence>\n\n"
        f"Log line:\n{line}\n"
    )

    text = ask_model(prompt)

    label_match = LABEL_RE.search(text)
    raw = label_match.group(1).upper() if label_match else "UNKNOWN"

    label = Label(raw) if raw in Label.__members__ else Label.UNKNOWN

    reason_match = REASON_RE.search(text)
    reason = reason_match.group(1).strip() if reason_match else text.strip()

    return Triage(line=line, label=label, reason=reason)

In [9]:
print(triage_log("Failed password for root from 203.0.113.9 port 4022 ssh2"))

Triage(line='Failed password for root from 203.0.113.9 port 4022 ssh2', label=<Label.SUSPICIOUS: 'SUSPICIOUS'>, reason='A failed root login attempt over SSH from an external IP address is a common indicator of brute-force or credential-stuffing attacks, warranting investigation.')


In [10]:
TOOL_NAME = "LogHawk"
print("Tool name:", TOOL_NAME)

Tool name: LogHawk


In [11]:
LOG_FILE = "security_log.csv"
MAX_ENTRIES = 20

def read_log_lines(path, limit=MAX_ENTRIES):
    with open(path, encoding="utf-8", errors="ignore") as f:
        lines = [ln.strip() for ln in f if ln.strip()]
    return lines[:limit]

entries = read_log_lines(LOG_FILE)
results = [triage_log(line) for line in entries]

print(f"Assessed {len(results)} entries from {LOG_FILE}")

Assessed 20 entries from security_log.csv


In [12]:
from datetime import datetime

def write_report(results, tool_name, path="assessment.md"):
    with open(path, "w", encoding="utf-8") as f:
        f.write(f"# {tool_name} Security Log Assessment\n\n")
        f.write(f"Generated: {datetime.now():%Y-%m-%d %H:%M}\n\n")
        f.write(f"Entries assessed: {len(results)}\n\n")

        for r in results:
            f.write(f"- **{r.label.value}** {r.reason}\n")
            f.write(f"  - `{r.line}`\n")

    return path

report_path = write_report(results, TOOL_NAME)

print("Wrote", report_path)
print()
print(open(report_path, encoding="utf-8").read())

Wrote assessment.md

# LogHawk Security Log Assessment

Generated: 2026-11-12 14:57

Entries assessed: 20

- **BENIGN** This is a CSV header line containing generic metadata fields with no actual log data or security-relevant content.
  - `﻿Keywords,Date and Time,Source,Event ID,Task Category`
- **BENIGN** Event ID 4672 indicates a normal special logon assignment (e.g., for system or high-privilege accounts) and is expected behavior in Windows security auditing when a privileged user logs on.
  - `Audit Success,11/12/2026 2:39:54 PM,Microsoft-Windows-Security-Auditing,4672,Special Logon,"Special privileges assigned to new logon.`
- **SUSPICIOUS** The log line is incomplete, containing only a header with no actual event data, which may indicate a malformed entry or potential log tampering attempt requiring investigation.
  - `Subject:`
- **BENIGN** The log line only indicates the SYSTEM account context, which is a standard and expected part of Windows security logging for system-level o